In [1]:
import os
# Change working root directory
os.chdir("/data-lachesis/common/propp/DeniseAtzori")

In [7]:
propp_outputs_directory = "/data-lachesis/common/propp/DeniseAtzori/BOOK_JEUNESSE"

In [3]:
#! pip install booknlp_fr -U
from propp_fr import (
    load_tokenizer_and_embedding_model,
    get_embedding_tensor_from_tokens_df,
    load_text_file,
    load_tokens_df,
    save_tokens_df,
    load_entities_df,
)
from tqdm.auto import tqdm
import torch
import os

/data-lachesis/common/propp/.venv/lib/python3.11/site-packages/propp_fr/propp_fr_add_entities_features.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


propp_fr package loaded successfully.


In [4]:
def extract_char_atts(row):
    result = []
    if row['char_att_agent'] != -1:
        result.append('agent')
    if row['char_att_patient'] != -1:
        result.append('patient')
    if row['char_att_mod'] != -1:
        result.append('mod')
    if row['char_att_poss'] != -1:
        result.append('poss')
    return result

In [5]:
tokenizer, embedding_model = load_tokenizer_and_embedding_model(model_name='almanach/camembert-large')

Some weights of CamembertModel were not initialized from the model checkpoint at almanach/camembert-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Loaded encoder model: almanach/camembert-large


In [13]:
import datetime

In [ ]:
embeddings_directory = propp_outputs_directory.replace("BOOK_JEUNESSE", "attribute_embeddings")
if not os.path.exists(embeddings_directory):
    os.makedirs(embeddings_directory)

extension = ".book"
book_files = sorted([f.replace(extension, "") for f in os.listdir(propp_outputs_directory) if f.endswith(extension)], reverse=False)

print(book_files)

#file_log = open(str(datetime.datetime.now()), 'w')

for file_name in tqdm(book_files[:]):
    try:
        print(file_name)
        attribute_embeddings_tensor_path = os.path.join(embeddings_directory, f"{file_name}.attribute_embeddings")

        if not os.path.exists(attribute_embeddings_tensor_path):
            print("Creando il file")

            txt_content = load_text_file(file_name, propp_outputs_directory)
            tokens_df = load_tokens_df(file_name, propp_outputs_directory)
            entities_df = load_entities_df(file_name, propp_outputs_directory)
            print("Robe caricate")
            tokens_embedding_tensor = get_embedding_tensor_from_tokens_df(txt_content,
                                                                        tokens_df,
                                                                        tokenizer,
                                                                        embedding_model,
                                                                        sliding_window_size='max',
                                                                        mini_batch_size= 16,
                                                                        sliding_window_overlap=0.5,
                                                                        subword_pooling_strategy="first_last")
            print("Tensore creato")
            att_cols = ["char_att_agent", "char_att_patient", "char_att_mod", "char_att_poss"]
            attributes_tokens_df = tokens_df[tokens_df[att_cols].ne(-1).any(axis=1)].copy()
            attributes_tokens_df['booknlp_categories'] = attributes_tokens_df.apply(extract_char_atts, axis=1)

            PER_entities_df = entities_df[entities_df["cat"] == "PER"]
            PER_entities_head_ids = PER_entities_df["head_id"].tolist()
            attributes_tokens_df = attributes_tokens_df[(attributes_tokens_df["char_att_agent"].isin(PER_entities_head_ids))
                                                        | (attributes_tokens_df["char_att_patient"].isin(PER_entities_head_ids))
                                                        | (attributes_tokens_df["char_att_mod"].isin(PER_entities_head_ids))
                                                        | (attributes_tokens_df["char_att_poss"].isin(PER_entities_head_ids))]

            print("Attributi caricati")
            attributes_indexes = attributes_tokens_df.index.tolist()

            tokens_df["is_PER_attribute"] = 0
            tokens_df.loc[attributes_indexes, "is_PER_attribute"] = 1
            save_tokens_df(tokens_df,file_name,propp_outputs_directory)

            attribute_embeddings = tokens_embedding_tensor[attributes_indexes]
            torch.save(attribute_embeddings, attribute_embeddings_tensor_path)

    except Exception as e: 
        print(e)
        #file_log.write(file_name)

['1833_Girardin-Delphine-de_Contes-d-une-vieille-fille-a-ses-neveux', '1835_Woillez-Catherine_Le-Robinson-des-demoiselles', '1843_Desnoyers-Louis_Les-aventures-de-Jean-Paul-Choppart', '1843_Woillez-Catherine_Leontine-et-Marie-ou-les-Deux-educations', '1845_Dumas-Alexandre_Histoire-d-un-Casse-noisette', '1846_Musset-Paul-de_Monsieur-le-Vent-et-Madame-la-Pluie', '1848_Woillez-Catherine_Edma-et-Marguerite-ou-les-Ruines-de-Chatillon-d-Azergues', '1851_Sand-George_Histoire-du-veritable-Gribouille', '1852_Carraud-Zulma-Tourangin-Mme_La-petite-Jeanne', '1854_Bassanville-Anais-de_Les-Primeurs-de-la-vie-ou-Bonheurs-joies-et-douleurs-de-la-jeunesse', '1854_Dumas-Alexandre_La-jeunesse-de-Pierrot', '1855_Solignac-Armand-de_Le-Gateau-des-rois-souvenirs-d-enfance', '1858_Gouraud-Julie_Les-Vacances-d-Yvonne', '1858_Pressense-Elise-de_Rosa', '1858_Segur-comtesse-de_Les-Malheurs-de-Sophie', '1858_Segur-comtesse-de_Les-Petites-Filles-Modeles', '1859_Segur-comtesse-de_Les-Vacances', '1860_Sauquet-Aricie-

  0%|          | 0/125 [00:00<?, ?it/s]

1833_Girardin-Delphine-de_Contes-d-une-vieille-fille-a-ses-neveux
Creando il file
Robe caricate


Embedding Tokens:   0%|          | 0/261 [00:00<?, ?it/s]

Tensore creato
Attributi caricati
name 'contatore' is not defined
1835_Woillez-Catherine_Le-Robinson-des-demoiselles
Creando il file
Robe caricate


...

Tensore creato
Attributi caricati
name 'contatore' is not defined
1930_Genestoux-Magdeleine-du_Mademoiselle-trouble-fete
Creando il file
Robe caricate


Embedding Tokens:   0%|          | 0/391 [00:00<?, ?it/s]

Tensore creato
Attributi caricati
name 'contatore' is not defined
1943_Saint-Exupery-Antoine-de_Le-petit-prince
Creando il file
Robe caricate


Embedding Tokens:   0%|          | 0/84 [00:00<?, ?it/s]

Tensore creato
Attributi caricati
name 'contatore' is not defined
